In [ ]:
using Apr9Project
using Plots

# Convergence Testing

## Forward Euler

In [ ]:
N_vals = [10, 100, 1000, 10000, 100000, 10^6]; # linearly spaced on logarithmic scale
h_vals = 1 ./ N_vals; # step sizes
errors = zeros(length(N_vals)); # to store errors

# set common parameters
f = (t,u) -> u;
u0 = 1.0
t0 = 0.0;
tmax = 1.0;

# enumerate is an iterrator that gives both the index and the value of each element in N_vals
for (i, N) in enumerate(N_vals)
    t_vals, u_vals = explicit_euler(f, u0, t0, tmax, N);
    errors[i] = abs(u_vals[end] - exp(1));
end


In [ ]:
errors

In [ ]:
c = exp(sum(log.(errors ./ h_vals)) / length(h_vals))
scatter(h_vals, errors, label="Data", xscale=:log10, yscale=:log10, legend=:topleft)
plot!(h_vals, c .* h_vals, label="Best-fit O(h)", linestyle=:dash)
xlabel!("h")
ylabel!("Error")
title!("Error vs Step Size")
xticks!(10.0.^(-6:1:0))
yticks!(10.0.^(-6:1:0))

Empirically, reducing $h$ by a factor of 10 reduces the error by a factor of 10.  This is $\mathrm{O}(h^1)$ behavior.  This is satisfactory empirical analysis of Forward Euler.

## Backwards Euler

In [ ]:
N_vals = [10, 100, 1000, 10000, 100000, 10^6]; # linearly spaced on logarithmic scale
h_vals = 1 ./ N_vals; # step sizes
errors = zeros(length(N_vals)); # to store errors

# set common parameters
f = (t,u) -> u;
u0 = 1.0
t0 = 0.0;
tmax = 1.0;

# enumerate is an iterrator that gives both the index and the value of each element in N_vals
for (i, N) in enumerate(N_vals)
    t_vals, u_vals = implicit_euler(f, u0, t0, tmax, N);
    errors[i] = abs(u_vals[end] - exp(1));
end

c = exp(sum(log.(errors ./ h_vals)) / length(h_vals))
scatter(h_vals, errors, label="Data", xscale=:log10, yscale=:log10, legend=:topleft)
plot!(h_vals, c .* h_vals, label="Best-fit O(h)", linestyle=:dash)
xlabel!("h")
ylabel!("Error")
title!("Error vs Step Size for Implicit Euler")
xticks!(10.0.^(-6:1:0))
yticks!(10.0.^(-6:1:0))


# Computational Cost of Explicit vs. Implicit

For `@btime`:

In [ ]:
using BenchmarkTools

Set up a problem and solve it two ways, timing the results.

In [ ]:
# set common parameters
f = (t,u) -> u;
u0 = 1.0
t0 = 0.0;
tmax = 1.0;
N = 10^3;

@btime explicit_euler($f, $u0, $t0, $tmax, $N);
@btime implicit_euler($f, $u0, $t0, $tmax, $N);

Note, use `$` for the arguments of functions that wish to time.  So if you want to time `f(x)`, run `@btime f($x)`.

In [ ]:
# set common parameters
f = (t,u) -> u;
u0 = 1.0
t0 = 0.0;
tmax = 1.0;
N = 10^4;

@btime explicit_euler($f, $u0, $t0, $tmax, $N);
@btime implicit_euler($f, $u0, $t0, $tmax, $N);

In [ ]:
# set common parameters
f = (t,u) -> u;
u0 = 1.0
t0 = 0.0;
tmax = 1.0;
N = 10^5;

@btime explicit_euler($f, $u0, $t0, $tmax, $N);
@btime implicit_euler($f, $u0, $t0, $tmax, $N);

Two empirical observations:
* Computational time for both methods scales linearly with `N`
* Implicit Euler, at least as implemented, is at least an order of magnitude slower
* Implicit Euler uses a lot more memory

# Root Finding
Solving
$$
f(x) = 0
$$
for $f:\mathbb{R}^1\to \mathbb{R}^1$.

In [ ]:
using Roots

## Example 1
Compute $\sqrt{2}$ by solving
$$
x^2 -2 =0
$$

In [ ]:
f=x-> x^2 - 2;
# our starting guess is 1.0, the second argument
root = find_zero(f, 1.0)

In [ ]:
sqrt(2)

## Example 2
Solve
$$
\cos(x) = x
$$
by doing root finding on $f(x)= \cos(x) - x$:


In [ ]:
f=x-> cos(x) - x;
# our starting guess is 1.0, the second argument
root = find_zero(f, 1.0)

In [ ]:
cos(0.7390851332151607)-0.7390851332151607

This also works with `f` defined as `function f(x)...`

What algorithm is this running?

In [ ]:
find_zero(f, 1.0, verbose=true)

We are not doing Newton, we are doing Secant (Newton-like).  The following code runs genuine Newton, but requires specification of the derviative function:

In [ ]:
f=x-> cos(x) - x;
df = x -> -sin(x) - 1;
# our starting guess is 1.0, the second argument
root = find_zero((f,df), 1.0, Roots.Newton(), verbose=true)